# 02b · Train, export, run: a spiking network end to end

Train a spiking network in snnTorch to turn random input spikes into a target spike pattern, export it to NIR, run it in py-spinnaker2 and compare. The target is the SpiNNaker2 logo, drawn as a raster plot, so the result can be judged by eye. Runs on Brian2; cluster participants run it on the chip in section 6.

Adapted from the vendor's example ["End to End NIR deployment"](https://gitlab.com/spinnaker2/py-spinnaker2/-/blob/v0.8.2/tutorials/nir/end_to_end_deployment/end_to_end_NIR.ipynb) (SpiNNcloud, Apache-2.0). Needs `snntorch==0.9.4`: newer versions break the NIR export. Each section ends with **Go deeper**: where to read more. Collapsed blocks (▶) hold background for anyone who wants more; the exercises don't need them.

**Workshop notes:** the folder [`01_knowledge_base`](../01_knowledge_base/) collects background written while preparing this workshop (spiking networks, the hardware, energy), with figures and further links. It is a reading collection for anyone who wants to delve deeper, not a primary source.

## 0. Setup

In [ ]:
import time

import matplotlib.pyplot as plt
import nir
import numpy as np
import snntorch as snn
import torch
import torch.nn as nn
import torch.nn.functional as F
from snntorch import surrogate
from snntorch.export_nir import export_to_nir
from spinnaker2 import brian2_sim, helpers, s2_nir

USE_HARDWARE = False


def backend():
    """A fresh backend for every run: Brian2, or one 48-chip SpiNNaker2 board."""
    if USE_HARDWARE:
        from spinnaker2 import hardware

        return hardware.SpiNNcloud48NodeBoard()
    return brian2_sim.Brian2Backend()


def raster(ax, matrix, title, color="royalblue"):
    """Raster plot of a spike matrix (neurons x time steps)."""
    neuron, t = np.nonzero(matrix)
    ax.scatter(t, neuron, s=0.5, color=color)
    ax.set(xlabel="time step", ylabel="neuron", title=title, xlim=(0, matrix.shape[1]), ylim=(matrix.shape[0], 0))

## 1. Input and target

Input: 256 neurons firing at random (Poisson, 40 spikes per second at 1 ms per step) for 300 steps. Target: 256 output neurons whose spikes draw the logo.

In [ ]:
n_neurons, steps = 256, 300
rng = np.random.default_rng(seed=0)
input_spikes = (rng.random((n_neurons, steps)) < 0.04).astype(np.float32)  # 0.04 = 40 Hz x 1 ms
target = np.load("data/scattered_logo.npy").astype(np.float32)  # shape (256, 300)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
raster(a1, input_spikes, "input: random spikes")
raster(a2, target, "target: the SpiNNaker2 logo")
plt.tight_layout()
plt.show()

## 2. The network

256 inputs → 512 LIF neurons → 256 LIF output neurons. Thresholds and decay factors are learnable per neuron. The surrogate gradient (`fast_sigmoid`) replaces the zero derivative of the spike function during training.

<details><summary>Surrogate gradients in short</summary>

A spike is a step function of the membrane potential, $S = H(U - \theta)$. Its derivative is 0 everywhere except at the threshold, where it is infinite, so backpropagation gets no useful gradient. The trick: keep the step function in the forward pass, but use the derivative of a smooth function in the backward pass. `fast_sigmoid` uses

$$\frac{\partial S}{\partial U} \approx \frac{1}{(1 + k\,|U - \theta|)^2}$$

with slope $k = 25$ by default: large near the threshold, small far from it. The network still produces real spikes, and gradients flow as if the step were a steep sigmoid. Review: [Neftci, Mostafa and Zenke 2019, IEEE Signal Processing Magazine](https://doi.org/10.1109/MSP.2019.2931595); hands-on: [snnTorch tutorial 5](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_5.html) and [tutorial 6](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_6.html).

</details>

In [ ]:
class LogoNet(nn.Module):
    def __init__(self, beta=0.5, threshold=0.3):
        super().__init__()
        grad = surrogate.fast_sigmoid()
        self.fc1 = nn.Linear(256, 512, bias=False)
        self.lif1 = snn.Leaky(beta=np.ones(512) * beta, threshold=np.ones(512) * threshold, spike_grad=grad,
                              reset_mechanism="zero", learn_threshold=True, learn_beta=True)
        self.fc2 = nn.Linear(512, 256, bias=False)
        self.lif2 = snn.Leaky(beta=np.ones(256) * beta, threshold=np.ones(256) * threshold, spike_grad=grad,
                              reset_mechanism="zero", learn_threshold=True, learn_beta=True)

    def forward(self, x):
        """x has shape (time, 1, 256); returns output spikes of shape (time, 1, 256)."""
        self.lif1.reset_mem()
        self.lif2.reset_mem()
        out = []
        for x_t in x:  # loops over the first dimension
            spk1, _ = self.lif1(self.fc1(x_t))
            spk2, _ = self.lif2(self.fc2(spk1))
            out.append(spk2)
        return torch.stack(out)


x = torch.tensor(input_spikes.T).unsqueeze(1)  # (time, 1, neurons)
y = torch.tensor(target.T).unsqueeze(1)
model = LogoNet()

## 3. Training

6,000 training steps of Adam on the mean squared error between output and target spikes. Takes about 3 minutes on a laptop CPU; reduce `epochs` if it is too slow.

**Note:** as in the original example, training treats the 300 time steps as independent samples, without memory across time. Evaluation and the chip run the network step by step, with memory, so the result is close to the target but not exact.

<details><summary>Why the training has no memory across time</summary>

The training passes the whole sequence as one block of shape `(1, time, 1, 256)`. The loop in `forward` runs over the first dimension, which here has length 1, so all 300 time steps go through the LIF layers **at once, as a batch of 300 single steps**, each starting from a membrane potential of 0. Training is fast this way, and since every output spike of the logo has to be produced from the input of the same step, it still works reasonably well.

Proper training of a spiking network runs it step by step over time, as the evaluation in section 4 does (input of shape `(time, 1, 256)`), and backpropagates through all steps: [backpropagation through time](https://en.wikipedia.org/wiki/Backpropagation_through_time), the same as for a recurrent network. That lets the network use its memory, at the cost of 300 sequential steps per training iteration. [snnTorch tutorial 5](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_5.html) shows this loop.

</details>

In [ ]:
epochs = 6000
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
batch_x, batch_y = x.unsqueeze(0), y.unsqueeze(0)  # (1, time, 1, neurons), as in the original example

start = time.time()
for epoch in range(1, epochs + 1):
    out = model(batch_x)
    loss = F.mse_loss(out, batch_y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 1000 == 0:
        print(f"epoch {epoch:5d}  loss {loss.item():.5f}  ({time.time() - start:.0f} s)")

## 4. Evaluate in snnTorch, export to NIR

Evaluation runs step by step over time (input of shape `(time, 1, 256)`). Then the trained model is exported to NIR and saved; a `.nir` file can be loaded on any machine with `nir.read`.

In [ ]:
with torch.no_grad():
    torch_out = model(x).squeeze(1).numpy().T  # (neurons, time)

nir_graph = export_to_nir(model.cpu(), x)
nir.write("logo_network.nir", nir_graph)
print("saved logo_network.nir")

## 5. Convert and run in py-spinnaker2

The conversion rule for snnTorch models applies (notebook 02a): `dt = 1e-4`, `FORWARD`, reset `ZERO`. The fully connected layers need a lot of synapse memory, so each core gets at most 30 LIF neurons.

<details><summary>Why at most 30 neurons per core</summary>

A core stores the incoming synapses of the neurons it simulates in its local memory. In the hidden layer, every neuron has 256 incoming synapses, in the output layer 512. With too many neurons per core, the synapses may not fit into a core's memory. `set_max_atoms_per_core(30)` is the value of the original example; the network then takes 9 cores for the output layer and 18 for the hidden layer, a tiny fraction of the 7,104 cores of a board. More in tutorial 05 (debugging, "a core running out of memory") and tutorial 08, section 4; the memory of a core in the workshop notes: [memory](../01_knowledge_base/02_hardware_energy.md#3-memory) and [when a network doesn't fit](../01_knowledge_base/02_hardware_energy.md#7-limits-in-practice-when-a-network-doesnt-fit).

</details>

In [ ]:
def run_spinnaker2():
    cfg = s2_nir.ConversionConfig()
    cfg.dt = 1e-4
    cfg.integrator = s2_nir.IntegratorMethod.FORWARD
    cfg.reset = s2_nir.ResetMethod.ZERO
    cfg.output_record = ["spikes"]
    cfg.conn_delay = 0
    cfg.scale_weights = True
    cfg.weight_scale_percentile = 100
    net, inp, outp = s2_nir.from_nir(nir.read("logo_network.nir"), cfg)
    for pop in net.populations:
        pop.set_max_atoms_per_core(100 if pop in inp else 30)
    inp[0].params = {i: list(np.nonzero(input_spikes[i])[0]) for i in range(n_neurons)}
    backend().run(net, steps + 2)  # 2 extra steps for the delays of the two projections
    m = helpers.spike_times_to_matrix(outp[0].get_spikes(), n_timesteps=steps + 2)
    return m[:, 2:]  # undo the chip's delay of one step per projection


def overlap(a, b):
    """Fraction of spikes in `a` that `b` has at the same neuron and step."""
    return (a * b).sum() / max(a.sum(), 1)


s2_out = run_spinnaker2()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
raster(axes[0], target, "target")
raster(axes[1], torch_out, "snnTorch (step by step)", color="darkorange")
raster(axes[2], s2_out, "py-spinnaker2", color="darkgreen")
plt.tight_layout()
plt.show()
print(f"py-spinnaker2 spikes on the target: {overlap(s2_out, target):.0%}")
print(f"snnTorch spikes reproduced by py-spinnaker2: {overlap(torch_out, s2_out):.0%}")

**Try:**

1. Train with `epochs = 1000` (section 3) and run sections 4 and 5 again. How much worse is the logo?
2. Why does py-spinnaker2 not reproduce the snnTorch output exactly here, while it did in notebook 02a? *Hint:* the weights are scaled to 8-bit integers (`scale_weights`), the thresholds with them.

**Discuss:** the network was trained without memory across time, but runs with memory on the chip. For which tasks would that shortcut fail?

**Go deeper:** the original example, with animations of input and output: ["End to End NIR deployment"](https://gitlab.com/spinnaker2/py-spinnaker2/-/blob/v0.8.2/tutorials/nir/end_to_end_deployment/end_to_end_NIR.ipynb). Training spiking networks step by step: [snnTorch tutorials](https://snntorch.readthedocs.io/en/latest/tutorials/index.html) 5–7. A larger trained network on the chip: the event-camera digits of the instructor demo, based on the [NMNIST example](https://spinnaker2.gitlab.io/py-spinnaker2/examples/nir_demo.html) of py-spinnaker2. Workshop notes: [SNNs and deep learning](../01_knowledge_base/01_primer_snn_basics.md#8-snns-and-deep-learning), with figures of the surrogate gradient and the deployment path.

## 6. Run on the SpiNNaker2 chip (cluster participants)

Set `USE_HARDWARE = True` and run the cell. The network uses about 30 cores on one 48-chip board; the run takes about one minute.

In [ ]:
USE_HARDWARE = False  # set to True on the cluster

chip_out = run_spinnaker2()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
raster(a1, target, "target")
raster(a2, chip_out, "SpiNNaker2 chip" if USE_HARDWARE else "Brian2 (USE_HARDWARE is False)", color="darkgreen")
plt.tight_layout()
plt.show()
print(f"spikes on the target: {overlap(chip_out, target):.0%}")
print(f"identical to the Brian2 run: {np.array_equal(chip_out, s2_out)}")